In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import FashionMNIST
from torchvision.utils import make_grid
from skimage.metrics import structural_similarity

TRAIN = True
SEED, BATCH = 42, 128
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CKPT_DIR, METRIC_DIR = ROOT / "results/checkpoints/custom_mlp_decoder", ROOT / "results/metrics/custom_mlp_decoder"
CKPT_DIR.mkdir(parents=True, exist_ok=True); METRIC_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED); np.random.seed(SEED)

train_raw = FashionMNIST(ROOT / "data", train=True, download=True)
test_raw = FashionMNIST(ROOT / "data", train=False, download=True)
images = torch.cat([train_raw.data, test_raw.data]).unsqueeze(1).float() / 255.0
labels = torch.cat([train_raw.targets, test_raw.targets])

with np.load(ROOT / "results/metrics/fashion_mnist_seed42.npz") as f:
    splits = {k: TensorDataset(images[f[k]], labels[f[k]]) for k in ("train", "val", "test")}

def get_loader(name):
    return DataLoader(splits[name], BATCH, shuffle=(name == "train"), generator=torch.Generator().manual_seed(SEED))

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1, self.fc2, self.fc3 = nn.Linear(784, 256), nn.Linear(256, 64), nn.Linear(64, 10)
    def features(self, x):
        return F.relu(self.fc2(F.relu(self.fc1(x.flatten(1)))))
    def forward(self, x):
        return self.fc3(self.features(x))

class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid(), nn.Unflatten(1, (1, 28, 28)))
    def forward(self, x):
        return self.decoder(self.encoder(x))

classifier = MLPClassifier().to(DEVICE)
decoder = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid(), nn.Unflatten(1, (1, 28, 28))).to(DEVICE)
vanilla_ae = Autoencoder().to(DEVICE)

if TRAIN:
    opt_c = torch.optim.Adam(classifier.parameters(), lr=1e-3)
    best_acc, hist_c = -1.0, []
    for epoch in range(1, 26):
        classifier.train()
        tr_ok, tr_n = 0, 0
        for x, y in get_loader("train"):
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = classifier(x)
            loss = F.cross_entropy(logits, y)
            opt_c.zero_grad(set_to_none=True); loss.backward(); opt_c.step()
            tr_ok += (logits.argmax(1) == y).sum().item(); tr_n += len(x)
        classifier.eval()
        with torch.no_grad():
            val_acc = sum((classifier(vx.to(DEVICE)).argmax(1) == vy.to(DEVICE)).sum().item() for vx, vy in get_loader("val")) / len(splits["val"])
        hist_c.append({"epoch": epoch, "train_acc": tr_ok / tr_n, "val_acc": val_acc})
        if val_acc > best_acc:
            best_acc, ep_c = val_acc, epoch
            best_wc = {k: v.cpu().clone() for k, v in classifier.state_dict().items()}
        if epoch % 5 == 0 or epoch == 1: print(f"[MLP] Epoch {epoch:02d}/25 | Val Acc: {val_acc:.4f}")
    classifier.load_state_dict({k: v.to(DEVICE) for k, v in best_wc.items()}); classifier.eval()
    for p in classifier.parameters(): p.requires_grad_(False)

    opt_d = torch.optim.Adam(decoder.parameters(), lr=1e-3)
    best_mse, hist_d = float("inf"), []
    for epoch in range(1, 31):
        decoder.train()
        tr_mse, tr_n = 0.0, 0
        for x, _ in get_loader("train"):
            x = x.to(DEVICE)
            with torch.no_grad(): z = classifier.features(x)
            loss = F.mse_loss(decoder(z), x)
            opt_d.zero_grad(set_to_none=True); loss.backward(); opt_d.step()
            tr_mse += loss.item() * len(x); tr_n += len(x)
        decoder.eval()
        with torch.no_grad():
            val_mse = sum(F.mse_loss(decoder(classifier.features(vx.to(DEVICE))), vx.to(DEVICE), reduction="sum").item() for vx, _ in get_loader("val")) / (len(splits["val"]) * 784)
        hist_d.append({"epoch": epoch, "train_mse": tr_mse / tr_n, "val_mse": val_mse})
        if val_mse < best_mse:
            best_mse, ep_d = val_mse, epoch
            best_wd = {k: v.cpu().clone() for k, v in decoder.state_dict().items()}
        if epoch % 5 == 0 or epoch == 1: print(f"[Decoder Beku] Epoch {epoch:02d}/30 | Val MSE (piksel^2): {val_mse:.6f}")
    decoder.load_state_dict({k: v.to(DEVICE) for k, v in best_wd.items()})
    torch.save({"ep_c": ep_c, "ep_d": ep_d, "c": best_wc, "d": best_wd}, CKPT_DIR / "mlp_decoder_same_run.pt")
    pd.DataFrame(hist_c).to_csv(METRIC_DIR / "mlp_history.csv", index=False)
    pd.DataFrame(hist_d).to_csv(METRIC_DIR / "decoder_history.csv", index=False)
else:
    ckpt = torch.load(CKPT_DIR / "mlp_decoder_same_run.pt", map_location="cpu", weights_only=True)
    classifier.load_state_dict(ckpt["c"]); decoder.load_state_dict(ckpt["d"]); ep_c, ep_d = ckpt["ep_c"], ckpt["ep_d"]

ae_ckpt = torch.load(ROOT / "results/checkpoints/custom_ae_vae/vanilla_ae_best.pt", map_location="cpu", weights_only=True)
vanilla_ae.load_state_dict(ae_ckpt["model_state_dict"])
classifier.eval(); decoder.eval(); vanilla_ae.eval()

In [ ]:
@torch.inference_mode()
def transpose_onehot(mlp, onehot):
    return torch.sigmoid(F.relu(F.relu(onehot @ mlp.fc3.weight) @ mlp.fc2.weight) @ mlp.fc1.weight).view(-1, 1, 28, 28)

@torch.inference_mode()
def transpose_latent(mlp, z):
    return torch.sigmoid(F.relu(z @ mlp.fc2.weight) @ mlp.fc1.weight).view(-1, 1, 28, 28)

def metrics(orig, est):
    mse = (orig - est).square().mean(dim=(1, 2, 3))
    psnr = (-10.0 * mse.clamp_min(1e-12).log10()).mean().item()
    ssim = float(np.mean([structural_similarity(a[0], b[0], data_range=1.0, win_size=7) for a, b in zip(orig.numpy(), est.numpy())]))
    return {"Dimensi Laten": 64, "MSE (piksel^2)": mse.mean().item(), "PSNR (dB)": psnr, "SSIM": ssim}

test_x, test_y = splits["test"].tensors
with torch.inference_mode():
    test_acc = (torch.cat([classifier(b.to(DEVICE)).cpu() for b in test_x.split(BATCH)]).argmax(1) == test_y).float().mean().item() * 100.0
    feats = torch.cat([classifier.features(b.to(DEVICE)) for b in test_x.split(BATCH)])
    rec_trans = torch.cat([transpose_latent(classifier, b).cpu() for b in feats.split(BATCH)])
    rec_dec = torch.cat([decoder(b).cpu() for b in feats.split(BATCH)])
    rec_ae = torch.cat([vanilla_ae(b.to(DEVICE)).cpu() for b in test_x.split(BATCH)])
    proto_trans = transpose_onehot(classifier, torch.eye(10, device=DEVICE)).cpu()
    proto_dec = decoder(F.relu(torch.eye(10, device=DEVICE) @ classifier.fc3.weight)).cpu()

    mu_z, std_z = feats.mean(0), feats.std(0).clamp_min(1e-6)
    rng = torch.Generator(device=DEVICE).manual_seed(SEED)
    gauss_samples = decoder(mu_z + std_z * torch.randn((10, 64), device=DEVICE, generator=rng)).cpu()

print(f"Akurasi Test MLP Classifier: {test_acc:.2f}% | Epoch MLP: {ep_c} | Epoch Decoder: {ep_d} | Vanilla AE Epoch: {ae_ckpt['epoch']}")
df = pd.DataFrame({"Proyeksi Transpos Laten MLP": metrics(test_x, rec_trans), "Decoder Beku (MLP 64-D)": metrics(test_x, rec_dec), "Vanilla AE (64-D)": metrics(test_x, rec_ae)}).T.round({"Dimensi Laten": 0, "MSE (piksel^2)": 6, "PSNR (dB)": 4, "SSIM": 4})
df.to_csv(METRIC_DIR / "mlp_decoder_comparison.csv")
display(df)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
cdf, ddf = pd.read_csv(METRIC_DIR / "mlp_history.csv"), pd.read_csv(METRIC_DIR / "decoder_history.csv")
axes[0].plot(cdf["epoch"], cdf["train_acc"], label="Train Acc"); axes[0].plot(cdf["epoch"], cdf["val_acc"], label="Val Acc"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(ddf["epoch"], ddf["train_mse"], label="Train MSE"); axes[1].plot(ddf["epoch"], ddf["val_mse"], label="Val MSE (piksel^2)"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

idx = [int((test_y == c).nonzero(as_tuple=True)[0][0]) for c in range(10)]
plt.figure(figsize=(12, 8)); plt.imshow(make_grid(torch.cat([proto_trans, proto_dec, gauss_samples, test_x[idx], rec_trans[idx], rec_dec[idx], rec_ae[idx]]), nrow=10, padding=2).permute(1, 2, 0).numpy(), cmap="gray"); plt.axis("off"); plt.show()